# Guía Práctica: Importación y Exportación de Archivos `.arff` en R

En este cuaderno aprenderemos a trabajar con archivos en formato **ARFF** (*Attribute-Relation File Format*), el estándar utilizado por herramientas de aprendizaje automático como **Weka**.

Utilizaremos como referencia la estructura clásica del dataset de Weka **`breast-cancer.arff`** y veremos dos enfoques para manipular estos archivos en **R**:
1. **Usando la librería estándar `foreign` (`read.arff` y `write.arff`)**.
2. **Escritura manual controlada** (muy útil cuando queremos garantizar que no aparezcan comillas innecesarias o problemas de formato en las cabeceras nominales).

## 1. Preparación: Crear un archivo `breast-cancer.arff` de ejemplo

Para que este cuaderno sea 100% autocontenido y puedas ejecutarlo directamente sin descargar archivos externos, primero crearemos en disco una muestra fiel del dataset original de Weka `breast-cancer.arff`.

In [ ]:
# Contenido representativo del dataset clásico breast-cancer.arff de Weka
contenido_arff <- "@relation breast-cancer

@attribute age {'10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90-99'}
@attribute menopause {'lt40','ge40','premeno'}
@attribute tumor-size {'0-4','5-9','10-14','15-19','20-24','25-29','30-34','35-39','40-44','45-49','50-54','55-59'}
@attribute inv-nodes {'0-2','3-5','6-8','9-11','12-14','15-17','18-20','21-23','24-26','27-29','30-32','33-35','36-39'}
@attribute node-caps {'yes','no'}
@attribute deg-malig {'1','2','3'}
@attribute breast {'left','right'}
@attribute breast-quad {'left_up','left_low','right_up','right_low','central'}
@attribute irradiat {'yes','no'}
@attribute Class {'no-recurrence-events','recurrence-events'}

@data
'40-49','premeno','15-19','0-2','yes','3','right','left_up','no','recurrence-events'
'50-59','ge40','15-19','0-2','no','1','right','central','no','no-recurrence-events'
'50-59','ge40','35-39','0-2','no','2','left','left_low','no','recurrence-events'
'40-49','premeno','35-39','0-2','yes','3','right','left_low','yes','no-recurrence-events'
'40-49','premeno','30-34','3-5','yes','2','left','right_up','no','recurrence-events'
'50-59','premeno','25-29','3-5','no','2','right','left_up','yes','no-recurrence-events'
'50-59','ge40','40-44','0-2','no','3','left','left_up','no','no-recurrence-events'
'40-49','premeno','10-14','0-2','no','2','left','left_up','no','no-recurrence-events'
'40-49','premeno','0-4','0-2','no','2','right','right_low','no','no-recurrence-events'
'40-49','ge40','40-44','15-17','yes','2','right','left_up','yes','no-recurrence-events'"

# Guardamos el archivo en el directorio de trabajo
writeLines(contenido_arff, "breast-cancer.arff")
cat("Archivo 'breast-cancer.arff' creado correctamente.\n")

## 2. Importación de un archivo `.arff` a un `data.frame` en R

El paquete **`foreign`** viene preinstalado en la mayoría de distribuciones de R y proporciona la función `read.arff()`, que convierte directamente los atributos nominales del archivo ARFF en variables de tipo `factor` en R.

*(Nota: Otra alternativa moderna disponible en CRAN es el paquete `farff`, que resulta más rápido con datasets muy grandes).* 

In [ ]:
# Cargamos la librería foreign
library(foreign)

# Importamos el archivo .arff
datos_cancer <- read.arff("breast-cancer.arff")

# Inspeccionamos la estructura del data.frame resultante
str(datos_cancer)

# Mostramos las primeras filas
head(datos_cancer)

### Comprobación de niveles y distribución de la clase

Al importar desde ARFF, R conserva todos los posibles valores definidos en `@attribute` como niveles (`levels`) del factor, incluso si alguno no aparece en las primeras filas.

In [ ]:
# Ver los niveles del atributo 'age'
cat("Niveles del atributo age:\n")
print(levels(datos_cancer$age))

# Tabla de frecuencias de la clase objetivo
cat("\nDistribución de la clase objetivo (Class):\n")
print(table(datos_cancer$Class))

## 3. Exportación de un `data.frame` a formato `.arff`

### Método A: Usando `write.arff()` del paquete `foreign`
Es la forma más directa. Es **fundamental** que las columnas categóricas estén definidas como `factor` en R antes de exportar; de lo contrario, `write.arff` podría tratarlas como cadenas de texto (`string`) en lugar de atributos nominales (`{valor1, valor2}`).

In [ ]:
# Filtramos, por ejemplo, solo los casos de pacientes con menopausia 'premeno'
subconjunto_premeno <- subset(datos_cancer, menopause == "premeno")

# Exportamos el subconjunto a un nuevo archivo .arff
write.arff(subconjunto_premeno, file = "breast-cancer-premeno.arff")

# Leemos las líneas del archivo generado para inspeccionar su sintaxis
cat(readLines("breast-cancer-premeno.arff"), sep = "\n")

### Método B: Exportación manual controlada (Recomendado para prácticas desde cero)

Cuando construimos un dataset desde cero (como en el juego de *Piedra, Papel, Tijera, Lagarto, Spock*), escribir la cabecera y los datos mediante conexión de archivo nos da control total sobre el nombre de `@relation` y el formato exacto de los atributos.

In [ ]:
# Creamos un data.frame de ejemplo simulando partidas
partidas <- data.frame(
  ronda = 1:4,
  jugada_usuario = factor(c("piedra", "spock", "lagarto", "tijera"), 
                          levels = c("piedra", "papel", "tijera", "lagarto", "spock")),
  jugada_maquina = factor(c("tijera", "papel", "lagarto", "lagarto"), 
                          levels = c("piedra", "papel", "tijera", "lagarto", "spock")),
  ganador = factor(c("usuario", "maquina", "empate", "usuario"), 
                   levels = c("usuario", "maquina", "empate"))
)

# Función personalizada para exportar con cabecera limpia
exportar_arff_limpio <- function(df, nombre_relacion, ruta_archivo) {
  con <- file(ruta_archivo, open = "wt", encoding = "UTF-8")
  
  # 1. Escribir relación
  writeLines(paste0("@relation ", nombre_relacion), con)
  writeLines("", con)
  
  # 2. Escribir atributos dinámicamente según el tipo de columna
  for (col_name in names(df)) {
    if (is.numeric(df[[col_name]])) {
      writeLines(paste0("@attribute ", col_name, " numeric"), con)
    } else if (is.factor(df[[col_name]])) {
      niveles <- paste(levels(df[[col_name]]), collapse = ", ")
      writeLines(paste0("@attribute ", col_name, " {", niveles, "}"), con)
    }
  }
  
  # 3. Escribir sección @data
  writeLines("", con)
  writeLines("@data", con)
  
  # Escribir filas separadas por comas sin nombres de fila ni comillas
  write.table(df, file = con, sep = ",", row.names = FALSE, col.names = FALSE, quote = FALSE)
  
  close(con)
}

# Ejecutamos la exportación
exportar_arff_limpio(partidas, "piedra_papel_tijera_lagarto_spock", "ejemplo_pptls.arff")

# Verificamos el resultado en pantalla y comprobamos que read.arff() lo lee sin errores
cat("--- CONTENIDO DEL ARCHIVO GENERADO ---\n")
cat(readLines("ejemplo_pptls.arff"), sep = "\n")

cat("\n--- VERIFICACIÓN DE LECTURA CON read.arff() ---\n")
verificacion <- read.arff("ejemplo_pptls.arff")
print(verificacion)